# <font color="steelblue">Proyecto no supervisado — Patrones de productos (Amazon Beauty) y comparación con la etiqueta</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a>

No olvides hacer una copia si deseas utilizarlo.

> **Guión de trabajo para el estudiante.** Este cuaderno es el **enunciado** de un proyecto de **aprendizaje no supervisado** que debéis **desarrollar vosotros**: objetivos, fases, tareas obligatorias, código, análisis de resultados conclusiones.

## <font color="steelblue">Objetivos del proyecto</font>

El dataset es, en origen, un problema de **recomendación**: 2 millones de **interacciones** usuario–producto con una valoración (`Rating` 1–5). Aquí lo abordamos de forma **no supervisada con una vuelta de tuerca**: en lugar de recomendar, vamos a

1. **Resumir cada producto** en un **perfil de comportamiento** (agregando sus valoraciones),
2. **agrupar los productos** por la similitud de ese perfil (clustering + reducción de dimensión),
3. **derivar una etiqueta** de "nivel de satisfacción" a partir del **Rating medio** de cada producto, y
4. **comparar la solución de cluster con esa etiqueta** mediante **matriz de contingencia, mapeo cluster→clase, índices externos y métricas ordinales**.

> **La etiqueta es CATEGÓRICA ORDINAL con 5 niveles (1–5)**, no numérica continua. Esto **afecta a las métricas**: los índices que tratan las clases como **nominales** (ARI, NMI) cuentan igual un error 4↔5 (leve) que 1↔5 (grave). Hay que **complementarlos** con métricas **sensibles al orden** (QWK, MAE ordinal, Spearman). Veamos dichas métricas:

  >* **QWK — Quadratic Weighted Kappa** (kappa cuadrática ponderada). Es la kappa de Cohen pero con una matriz de pesos que penaliza los errores según su distancia al cuadrado: $wij=(i-j)^2/(K-1)^2$, con K categorías. Así, equivocarse en una categoría adyacente penaliza poco y un salto grande penaliza mucho. Está corregida por azar (un acuerdo aleatorio da ≈0) y vale 1 en el acuerdo perfecto. Es el estándar de facto en tareas ordinales. En scikit-learn: cohen_kappa_score(y, y_pred, weights='quadratic') (con weights='linear' la penalización crece de forma lineal en vez de cuadrática).
  >* **MAE — Error Absoluto Medio (en versión ordinal)**. Codificas las categorías como enteros $1,…,K$ y calculas $\frac{1}{n}\sum_i |\hat{y}_i - y_i|$. Se interpreta directamente como «en promedio fallo *X* categorías de distancia». Es muy intuitivo y, con clases desbalanceadas, conviene su variante **MAE macro** (promedio del MAE por clase) para que las clases minoritarias no queden ocultas. En scikit-learn: `mean_absolute_error` sobre las etiquetas enteras.
  >* **Spearman** (ρ, correlación de rangos). Mide si el modelo ordena las muestras de forma coherente con la realidad (asociación monótona), sin exigir que acierte la categoría exacta. Va de −1 a 1. Es útil cuando lo que importa es el ranking (clasificar bien de menor a mayor) más que el valor concreto. En SciPy: scipy.stats.spearmanr.

> **Aviso metodológico (importante):** la etiqueta se **deriva** del propio `Rating` medio; no es una clase externa "real". Por tanto, la comparación es **ilustrativa** (validación externa con una etiqueta construida), no una verdad de referencia independiente. Lo interesante es ver **qué parte** de la estructura no supervisada se alinea con la satisfacción y **qué otras dimensiones** (popularidad, polarización, recencia) la **cruzan**.

> Reutilizad los cuadernos del curso en el bloque de técnicas no supervisadas apra resolver todas als cuestiones planteadas.

## <font color="steelblue">El conjunto de datos</font>

El conjunto de datos contiene **2.023.070 interacciones** (Amazon *Beauty*, "ratings only"; McAuley et al.). Sin valores perdidos. la base de datos está compuesta por cuatro columnas:

* **`UserId`** — identificador del usuario.
* **`ProductId`** — identificador del producto (ASIN).
* **`Rating`** — valoración entera **1–5**.
* **`Timestamp`** — momento (Unix, segundos) → `pd.to_datetime(..., unit='s')`.

**Naturaleza:** la información contenida en los datos es **muy dispersa** (la mayoría de productos/usuarios tienen **muy pocas** valoraciones; unos pocos, miles) y `Rating` **sesgado** hacia 5. Por eso te recomendamos trabajar con  **productos con suficientes valoraciones**, para que sus perfiles sean estables.

## <font color="steelblue">Reglas del juego (buenas prácticas obligatorias)</font>

1. **Elige la entidad a agrupar** (recomendado: **productos**) y **agrega** sus interacciones en un **perfil** numérico.
2. **Filtra** entidades con **muy pocas** valoraciones (p. ej. < 20–50): sus perfiles son ruido.
3. **Escala** antes de PCA/clustering.
4. **Deriva la etiqueta como ORDINAL 1–5** (redondeo o moda del rating del producto) y **resérvala** para la comparación; decide **con qué variables** clusterizas (incluir o no los estadísticos de rating tiene consecuencias).
6b. **Métricas sensibles al orden:** al comparar, usa índices nominales **y** ordinales (QWK/MAE/Spearman), porque la etiqueta es ordinal.
5. **Combina modelos:** reduce dimensión + agrupa, y compara varios algoritmos.
6. **Compara con la etiqueta** al final con `classification_report` + **ARI/NMI**, e **interpreta con cautela** (la etiqueta es derivada).
7. **Caracteriza y nombra** los grupos; **reproducibilidad** con `random_state`.

# <font color="steelblue">Fase 0 — Preparación del entorno y carga de datos</font>

In [ ]:
path = kagglehub.dataset_download("skillsmuggler/amazon-ratings")
archivos = os.listdir(path); print("Archivos:", archivos)
ratings = pd.read_csv(os.path.join(path, archivos[0]))
# Asegura nombres de columna (ajusta si difieren): UserId, ProductId, Rating, Timestamp
ratings.columns = ['UserId', 'ProductId', 'Rating', 'Timestamp'][:ratings.shape[1]]
print(f"Interacciones: {ratings.shape[0]:,} | columnas: {list(ratings.columns)}")
ratings.head()

# <font color="steelblue">Fase 1 — Comprensión y EDA</font>

**Tareas a desarrollar**
1. **Volumen y unicidad.** Nº de interacciones, de **usuarios** y de **productos** únicos.
2. **Distribución del `Rating`.** Histograma (verás fuerte **sesgo a 5**).
3. **Dispersión / cola larga.** Nº de valoraciones **por producto** y **por usuario**: la inmensa mayoría tiene muy pocas (distribución muy asimétrica). Justifica el **filtro**.
4. **Tiempo.** Convierte `Timestamp` a fecha y observa el **rango** (1996–2014) y la evolución del volumen.
5. **Entidad.** Decide agrupar **productos** (o usuarios) y por qué.
6. **Conclusión:** 3–4 hallazgos.

# <font color="steelblue">Fase 2 — Perfil por entidad, etiqueta derivada y escalado</font>

**Tareas a desarrollar**
1. **Agrega por producto** un **perfil** numérico, por ejemplo: `n_valoraciones`, `rating_medio`, `rating_desv`, **proporción de 1,2,3,4,5 estrellas** (forma de la distribución), y rasgos **temporales** (antigüedad, recencia respecto a 2014, valoraciones por año).
2. **Filtra** los productos con **pocas** valoraciones (p. ej. `n_valoraciones >= 30`) para perfiles estables.
3. **Etiqueta derivada ORDINAL (clave).** Crea `rating_ord` **categórica ordinal con niveles 1–5** a partir del rating del producto: por **redondeo** del `rating_medio` o por la **moda** de sus valoraciones. Declárala `ordered=True`. **Resérvala** para la Fase 6 (no la metas como variable de entrada salvo en el experimento que se indica).
   > Por el sesgo a 5, los niveles **1–2 serán escasos**: tenlo en cuenta al leer las métricas por clase.
4. **Escala** el perfil (StandardScaler). Conserva el perfil **sin escalar** para interpretar.

> **Decisión con consecuencias:** si clusterizas **incluyendo** los estadísticos de rating, la alineación con la etiqueta será en parte **trivial** (la etiqueta sale de ahí). Plantéate clusterizar con **dos conjuntos**: (A) perfil completo y (B) solo **actividad/temporal** (sin estadísticos de rating), y compara la alineación con la etiqueta en la Fase 6.

# <font color="steelblue">Fase 3 — Reducción de dimensión</font>

**Tareas a desarrollar**
1. **PCA** sobre el perfil escalado: **scree plot** e **interpretación de loadings** (¿qué mide PC1? ¿popularidad? ¿satisfacción? ¿polarización?).
2. **Visualización.** Proyecta en PC1–PC2 (y con **MDS**/**t-SNE** sobre una **muestra**, por coste) **coloreando por la etiqueta ordinal `rating_ord`**: ¿se separan los niveles 1–5? ¿hay otras direcciones de variación?
3. **Conclusión:** dimensionalidad efectiva y qué capturan las componentes.

> Colorear por la etiqueta aquí es un **anticipo visual** de la comparación de la Fase 6.

# <font color="steelblue">Fase 4 — Agrupación y combinación de modelos</font>

**Tareas a desarrollar**
1. Aplica y compara **≥3** algoritmos: **K-Means**, **jerárquico** (+ dendrograma) y **DBSCAN** y/o **GMM**.
2. **Combina con la reducción:** agrupa sobre el **espacio PCA** y compáralo con el perfil escalado.
3. Hazlo para los **dos conjuntos** de variables (A completo y B actividad/temporal) si has optado por el experimento.
4. **Concordancia** entre soluciones (Adjusted Rand Index).

> Para comparar con una etiqueta de 5 niveles puedes explorar `k≈5`, pero **no fuerces**: el nº de grupos se decide en la Fase 5 por criterios internos, y la comparación admite que clusters distintos mapeen al mismo nivel.

# <font color="steelblue">Fase 5 — Validación interna y número de grupos</font>

**Tareas a desarrollar**
1. **Número de grupos:** codo (inercia) y **silueta** variando `k`; dendrograma para el jerárquico.
2. **Índices internos:** silueta, **Davies–Bouldin**, **Calinski–Harabasz**.
3. **Estabilidad** ante semillas/submuestras.
4. **Decisión justificada** del algoritmo, el espacio (original/PCA) y el número de grupos.

# <font color="steelblue">Fase 6 — Comparación con la etiqueta ordinal (validación externa)</font>

Contrastar la **solución de cluster** con la **etiqueta ordinal** `rating_ord` (1–5). Como la etiqueta **tiene orden**, hay que usar métricas **nominales y ordinales**.

**Tareas a desarrollar**
1. **Matriz de contingencia** grupos × `rating_ord` (`pd.crosstab`): ¿cada cluster se asocia a un nivel? Obsérvala como una "confusión" y fíjate en la **distancia a la diagonal**.
2. **Índices externos nominales:** **Adjusted Rand Index** y **NMI** (no usan el orden: trátalos como referencia parcial).
3. **Mapeo cluster→nivel** (clase **mayoritaria** de cada cluster) → `y_pred` ordinal, y **`classification_report`** (recuerda: macro-F1 **ignora el orden**).
4. **Métricas ORDINALES (clave):** sobre `y_true`/`y_pred` numéricos 1–5 calcula
   * **QWK** — `cohen_kappa_score(..., weights='quadratic')`: penaliza **más** los errores lejanos (1↔5) que los cercanos (4↔5),
   * **MAE ordinal** — error medio en niveles,
   * **Spearman** — correlación de rangos.
5. **Experimento A vs B:** repite con perfil completo (A) y solo actividad/temporal (B); compara la alineación (QWK/MAE) y discute cuánta es **trivial**.
6. **Interpretación crítica:** la etiqueta es **derivada y ordinal**; un QWK/MAE razonable con ARI bajo indica que los clusters captan el **nivel** aunque no coincidan 1:1 con las clases.

> **Por qué importa el orden:** para una etiqueta ordinal, *accuracy*/macro-F1/ARI cuentan igual confundir 4 con 5 que 1 con 5. **QWK y MAE** sí distinguen la **gravedad** del error: son las métricas correctas aquí.

# <font color="steelblue">Fase 7 — Caracterización de los grupos</font>

**Tareas a desarrollar**
1. **Perfiles.** Media de cada variable del perfil (sobre valores **originales**) por grupo → **mapa de calor**.
2. **Rasgos distintivos.** Para cada grupo, qué lo define (z-score frente a la media global): popularidad, satisfacción, **polarización** (muchos 1 y 5), recencia…
3. **Nombre** de cada grupo (p. ej. "éxitos masivos bien valorados", "nicho polarizado", "tibios populares", "novedades poco valoradas"…) y **tamaño**.
4. **Lectura de negocio.** Utilidad para *merchandising*, control de calidad, detección de productos problemáticos.

# <font color="steelblue">Fase 8 — Aplicación interactiva (widget en Colab)</font>

Aplicación **dentro del cuaderno** con **`ipywidgets`** (no despliegue). Implementa **al menos una**:

1. **Explorador de grupos:** desplegable de grupo → perfil medio, tamaño, su nivel ordinal (1–5) dominante y productos de ejemplo.
2. **Buscador por producto:** elige un `ProductId` (de los filtrados) y muestra su perfil, su **grupo**, su nivel ordinal (1–5) y productos "parecidos" del mismo grupo.
3. **(Opcional)** Dispersión PC1–PC2 coloreada por grupo (o por etiqueta) con `plotly`.

> El widget solo **usa** el modelo ya ajustado; no reentrena en cada interacción.

# <font color="steelblue">Entregables y formato</font>

* **Cuaderno** ejecutable de principio a fin, con código y **celdas de texto** que justifiquen cada decisión (filtro, etiqueta derivada, nº de grupos, **interpretación de la comparación**).
* **Memoria breve:** EDA, construcción del perfil y la etiqueta, reducción de dimensión, comparación de algoritmos, validación interna, **comparación con la etiqueta (contingencia + `classification_report` + ARI/NMI)**, caracterización y la aplicación.
* **Widget(s)** funcionando dentro del cuaderno.
* **Reproducibilidad:** `random_state` fijado y comentario de estabilidad.

## <font color="steelblue">Rúbrica de evaluación (100 pts)</font>

| Fase | Qué se valora | Puntos |
|---|---|---:|
| 1. EDA | Escala, sparsity/cola larga, distribución de Rating, elección de entidad | 12 |
| 2. Perfil + etiqueta | Agregación, **filtro**, **etiqueta derivada del Rating medio**, escalado, conjuntos A/B | 14 |
| 3. Reducción de dimensión | PCA (interpretación) + MDS/t-SNE, color por etiqueta | 14 |
| 4. Clustering y combinación | ≥3 algoritmos, **clustering sobre PCA vs original (y A vs B)** | 14 |
| 5. Validación interna | Codo/silueta, índices, estabilidad | 10 |
| 6. **Comparación con la etiqueta ordinal** | Contingencia + ARI/NMI + mapeo cluster→clase + `classification_report` + **métricas ordinales (QWK/MAE/Spearman)**, experimento A/B, crítica | 18 |
| 7. Caracterización | Perfiles, rasgos, **nombres**, lectura de negocio | 12 |
| 8. Aplicación (widget) | `ipywidgets` funcional | 6 |
| — Extra | UMAP, consenso de clustering, usuarios como entidad, recomendador en el widget | +10 |

# <font color="steelblue">Pistas y errores típicos</font>

* **Filtra primero.** Productos con 1–2 valoraciones dan perfiles sin sentido y dominan por número.
* **La etiqueta es derivada.** Sale del `rating_medio`: si clusterizas con los estadísticos de rating, parte de la "coincidencia" es **tautológica**. Por eso el experimento **A vs B**.
* **Escala** antes de PCA/clustering; `n_valoraciones` tiene una escala enorme frente a las proporciones.
* **Validación externa ≠ supervisión.** Mapear cluster→clase mayoritaria + `classification_report` mide **alineación**, no entrena con la etiqueta.
* **Etiqueta ORDINAL:** acompaña *accuracy*/F1/ARI con **QWK y MAE**; un 4 tomado por 5 no es tan grave como un 1 tomado por 5, y solo las métricas ordinales lo reflejan.
* **Interpreta.** Un ARI bajo no es "fracaso": puede significar que la estructura real (popularidad, polarización, recencia) **no coincide** con la satisfacción media — un hallazgo en sí.
* **Widget, no despliegue:** `ipywidgets` dentro de Colab.

# <font color="steelblue">Referencias</font>

* McAuley, J. et al. (2015). *Image-based recommendations on styles and substitutes*. SIGIR '15.
* He, R. & McAuley, J. (2016). *Ups and Downs: ...One-Class Collaborative Filtering*. WWW '16.
* *Amazon - Ratings (Beauty Products)*. Kaggle.
* Cuadernos del curso: *Componentes principales y variantes*, *Escalas multidimensionales (MDS)*, *Clustering (K-Means, jerárquico, DBSCAN, mixturas gaussianas)*.